# Fato Cadastro

In [0]:
import pyspark.sql.functions as f

In [0]:
spark.sql('USE CATALOG saude_sus')

In [0]:
df_trusted = spark.read.table('saude_sus.trusted.tru_estabelecimento')
df_dim_calendario = spark.read.table('saude_sus.refined.DIM_CALENDARIO')
df_dim_localizacao = spark.read.table('saude_sus.refined.dim_localizacao')
df_dim_turno = spark.read.table('saude_sus.refined.DIM_TURNO')
df_dim_tipo_unidade = spark.read.table('saude_sus.refined.DIM_TIPO_UNIDADE')
df_dim_estabelecimento = spark.read.table('saude_sus.refined.dim_estabelecimento')

In [0]:
df_fact = df_trusted.alias('f') \
    .join(
        df_dim_localizacao.alias("l"),
        (f.col("f.COD_UF") == f.col("l.COD_UF")) & 
        (f.col("f.COD_MUNICIPIO") == f.col("l.COD_MUNICIPIO")) & 
        (f.upper(f.coalesce(f.col("f.END_BAIRRO_ESTABELECIMENTO"), f.lit("NAO INFORMADO"))) == f.col("l.END_BAIRRO_ESTABELECIMENTO")) & 
        (f.upper(f.coalesce(f.col("f.END_CEP_ESTABELECIMENTO"), f.lit("NAO INFORMADO"))) == f.col("l.END_CEP_ESTABELECIMENTO")) &
        (f.upper(f.coalesce(f.col("f.END_LOGRADOURO_ESTABELECIMENTO"), f.lit("NAO INFORMADO"))) == f.col("l.END_LOGRADOURO_ESTABELECIMENTO")),
        "left"
    ) \
    .join(
        df_dim_calendario.alias("c"),
        f.col("f.DAT_ATUALIZACAO") == f.col("c.DAT_REFERENCIA"),
        "left"
    ) \
    .join(
        df_dim_turno.alias("t"),
        f.col("f.COD_IDENTIFICADOR_TURNO_ATENDIMENTO") == f.col("t.COD_IDENTIFICADOR_TURNO_ATENDIMENTO"),
        "left"
    ) \
    .join(
        df_dim_tipo_unidade.alias('tp'),
        f.col('f.COD_TIPO_UNIDADE') == f.col('tp.COD_TIPO_UNIDADE'),
        "left"
    ) \
    .join(df_dim_estabelecimento.alias('e'), 
        (f.col('f.COD_CNES') == f.col('e.COD_CNES')) & 
        (f.col('f.DAT_ATUALIZACAO') >= f.col('e.DAT_INICIO_VIGENCIA')) & 
        (f.col('f.DAT_ATUALIZACAO') < f.coalesce(f.col('e.DAT_FIM_VIGENCIA'), f.lit('9999-12-31'))), 
        "left"
    ) \
    .select(
        # Keys
        f.coalesce(f.col("e.SK_ESTABELECIMENTO"), f.lit("-1")).alias("SK_ESTABELECIMENTO"),
        f.coalesce(f.col("l.SK_LOCALIZACAO"), f.lit("-1")).alias("SK_LOCALIZACAO"),
        f.coalesce(f.col("c.SK_CALENDARIO"), f.lit("-1")).alias("SK_CALENDARIO"),
        f.coalesce(f.col("t.SK_TURNO"), f.lit("-1")).alias("SK_TURNO"),
        f.coalesce(f.col("tp.SK_TIPO_UNIDADE"), f.lit("-1")).alias("SK_TIPO_UNIDADE"),

        # Aggregations
        f.coalesce(f.col("FLG_POSSUI_ATENDIMENTO_AMBULATORIAL_SUS").cast("int"), f.lit(0)).alias("QTD_ATENDIMENTO_SUS"),
        f.coalesce(f.col("FLG_POSSUI_CENTRO_CIRURGICO").cast("int"), f.lit(0)).alias("QTD_CENTRO_CIRURGICO"),
        f.coalesce(f.col("FLG_POSSUI_CENTRO_OBSTETRICO").cast("int"), f.lit(0)).alias("QTD_CENTRO_OBSTETRICO"),
        f.coalesce(f.col("FLG_POSSUI_CENTRO_NEONATAL").cast("int"), f.lit(0)).alias("QTD_CENTRO_NEONATAL"),
        f.coalesce(f.col("FLG_POSSUI_ATENDIMENTO_HOSPITALAR").cast("int"), f.lit(0)).alias("QTD_ATENDIMENTO_HOSPITALAR"),
        f.coalesce(f.col("FLG_POSSUI_SERVICO_APOIO").cast("int"), f.lit(0)).alias("QTD_SERVICO_APOIO"),
        f.coalesce(f.col("FLG_POSSUI_ATENDIMENTO_AMBULATORIAL").cast("int"), f.lit(0)).alias("QTD_ATENDIMENTO_AMBULATORIAL")
    )

In [0]:
df_fact.write.mode("append").saveAsTable('saude_sus.refined.fact_cadastro')